# Parquet 导出与回读核对

本课沿用第 22 课的九表入库清单。先用商品表练习写入和回读，再核对九张拟入库表的导出报告。


## 1. 选择项目虚拟环境

在 VS Code 的 Notebook 内核选择器中选择项目目录下的 .venv/Scripts/python.exe。当前项目虚拟环境基于 myenv 的 Python 3.8.20，Parquet 引擎使用 PyArrow 17.0.0。 [PyArrow 17.0.0 的 Windows Python 3.8 安装包](https://pypi.org/project/pyarrow/17.0.0/#pyarrow-17.0.0-cp38-cp38-win_amd64.whl)

在项目根目录的 PowerShell 终端运行以下命令，可以生成本课所用的九张 Parquet 表。

    .\.venv\Scripts\python.exe .\src\10_export_parquet.py


In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

import pandas as pd
from IPython.display import display

try:
    import pyarrow
except ImportError as exc:
    raise ImportError('请选择项目 .venv/Scripts/python.exe 作为 Notebook 内核') from exc

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from mysql_input import read_manifest, load_input_table

manifest = read_manifest()
table_specs = manifest['tables']
processed_dir = project_dir / 'data' / 'processed'

print('当前内核', sys.executable)
print('Pandas', pd.__version__)
print('PyArrow', pyarrow.__version__)


当前内核 d:\数据分析\olist-operations-analytics\.venv\Scripts\python.exe
Pandas 2.0.3
PyArrow 17.0.0


## 2. 用商品表练习写入和回读

先从 CSV 恢复字段类型，再写入临时 Parquet。回读后逐列比较数据、类型和空值。
商品表中的重量是否为零是可空布尔，重量未知时仍保留缺失。


In [5]:
products = load_input_table(processed_dir, 'products', table_specs['products'])
with TemporaryDirectory() as folder:
    demo_path = Path(folder) / 'products.parquet'
    products.to_parquet(demo_path, engine='pyarrow', compression='snappy', index=False)
    products_readback = pd.read_parquet(demo_path, engine='pyarrow')
    pd.testing.assert_frame_equal(products, products_readback, check_dtype=True, check_exact=True)

display(pd.DataFrame({
    '写入前类型': products.dtypes.astype(str),
    '回读后类型': products_readback.dtypes.astype(str),
    '回读缺失数': products_readback.isna().sum(),
}).loc[['product_id', 'product_weight_g', 'flag_weight_zero']])
display(products_readback['flag_weight_zero'].value_counts(dropna=False))


,写入前类型,回读后类型,回读缺失数
product_id,string,string,0
product_weight_g,Int64,Int64,2
flag_weight_zero,boolean,boolean,2


flag_weight_zero
False    32945
True         4
<NA>         2
Name: count, dtype: Int64

## 3. 检查九表导出报告

导出脚本会逐表回读，比较行数、类型、值和空值，再发布文件。核对表中的行数差额与主键重复数应为 0。


In [6]:
report_dir = project_dir / 'reports' / 'mysql_input_v1'
check_path = report_dir / 'parquet_roundtrip_v1.csv'
if not check_path.is_file():
    raise FileNotFoundError('请先在项目终端运行 src/10_export_parquet.py 生成导出报告')

parquet_checks = pd.read_csv(check_path, encoding='utf-8-sig')
display(parquet_checks)


,表,导出前行数,回读行数,行数差额,回读主键重复数,回读缺失单元格数,字段类型与值一致,文件字节数
0,customers,99441,99441,0,0,0,True,7112390
1,sellers,3095,3095,0,0,0,True,140016
2,category_translation,71,71,0,0,0,True,4528
3,products,32951,32951,0,0,2452,True,1512387
4,geo_reference,19019,19019,0,0,0,True,525026
5,orders,99441,99441,0,0,14005,True,12273098
6,order_items,112650,112650,0,0,0,True,6507772
7,order_payments,103886,103886,0,0,0,True,3864078
8,order_reviews,99224,99224,0,0,145903,True,10453979


## 4. 回读三项整数分金额

用两张正式导出表重新计算商品金额、运费和支付额，与入库清单的基准合计比较。


In [7]:
parquet_dir = processed_dir / 'mysql_input_v1'
amount_checks = []
for name in ['order_items', 'order_payments']:
    frame = pd.read_parquet(parquet_dir / (name + '.parquet'), engine='pyarrow')
    for column, expected in table_specs[name]['expected_totals'].items():
        actual = int(frame[column].sum())
        amount_checks.append({
            '字段': name + '.' + column,
            '回读合计（分）': actual,
            '差额（分）': actual - expected,
        })
        if actual != expected:
            raise ValueError('Parquet 回读金额与清单不符')
display(pd.DataFrame(amount_checks))


,字段,回读合计（分）,差额（分）
0,order_items.price_cents,1359164370,0
1,order_items.freight_cents,225190954,0
2,order_payments.payment_value_cents,1600887212,0


## 我的理解

请发来第 3 格的九表核对表。解释为什么商品重量未知时，flag_weight_zero 应该保留空值，以及回读 Parquet 时可以用哪一列来验证这个规则。
商品重量未知时，我们无法判断重量是否为零，所以 flag_weight_zero 应保留空值,几位true or false都会计入数量。回读 Parquet 就是把刚保存的文件重新读入 Pandas。回读后，我先用 product_weight_g 找出重量为空的两条记录，再检查这两条记录的 flag_weight_zero 是否也为空。`product_weight_g` 为空时，对应的 flag_weight_zero 也应为空，因为重量未知，无法判断它是否为零。
核对完成后，按同一份清单准备 MySQL 建表。
用 Parquet，是为了把清洗后的数据连同字段类型一起保存，作为 MySQL 入库前的中间文件。CSV 主要保存文本，每次读入都要按清单恢复类型。例如邮编 01310 如果被读成整数，就会变成 1310。在本项目的 Parquet 回读中，邮编仍是字符串，金额仍是整数分，布尔标记的空值也保留了。